In [1]:
# Configuration
API_KEY = "GMb61lvkA3ZDdwnwE0XcGyHKWGKMXH14OfRa6hKH"
ENDPOINT = "https://api.nasa.gov/neo/rest/v1/feed"
PARAMS = {
    "start_date": "2026-03-01",
    "end_date": "2026-03-07",
    "api_key": API_KEY
}

Ingestion (Bronze)

In [9]:
import subprocess
import json
from datetime import datetime
import os



In [16]:
def ingest_eonet():
    """Fetch NASA EONET events and save raw JSON to Bronze."""
    url = (
        "https://eonet.gsfc.nasa.gov/api/v3/events"
        "?status=all&start=2024-01-01&end=2024-12-31"
        "&category=wildfires,severeStorms"
    )

    print("Fetching NASA EONET events...")
    result = subprocess.run(
        ["curl", "-s", "--max-time", "120", url],
        capture_output=True, text=True
    )

    if result.returncode != 0:
        raise RuntimeError(f"curl failed: {result.stderr}")

    data = json.loads(result.stdout)
    event_count = len(data.get("events", []))
    print(f"Retrieved {event_count} events")

    os.makedirs("data/bronze/eonet", exist_ok=True)
    ts = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
    filename = f"data/bronze/eonet/eonet_events_{ts}.json"

    with open(filename, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Saved {filename}")
    return filename


if __name__ == "__main__":
    ingest_eonet()


Fetching NASA EONET events...
Retrieved 5700 events
Saved data/bronze/eonet/eonet_events_2026-04-01T22-31-54.json


In [12]:
import subprocess
import json
from datetime import datetime
import os


In [15]:
def ingest_weather():
    """Fetch Open-Meteo historical weather and save raw JSON to Bronze."""
    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        "?latitude=34.05&longitude=-118.24"
        "&start_date=2024-01-01&end_date=2024-12-31"
        "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max"
        "&timezone=America/Los_Angeles"
    )

    print("Fetching Open-Meteo historical weather for Los Angeles...")
    result = subprocess.run(
        ["curl", "-s", "--max-time", "60", url],
        capture_output=True, text=True
    )

    if result.returncode != 0:
        raise RuntimeError(f"curl failed: {result.stderr}")

    data = json.loads(result.stdout)
    day_count = len(data.get("daily", {}).get("time", []))
    print(f"Retrieved {day_count} days of weather data")

    os.makedirs("data/bronze/weather", exist_ok=True)
    ts = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
    filename = f"data/bronze/weather/la_weather_{ts}.json"

    with open(filename, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Saved {filename}")
    return filename


if __name__ == "__main__":
    ingest_weather()



Fetching Open-Meteo historical weather for Los Angeles...
Retrieved 366 days of weather data
Saved data/bronze/weather/la_weather_2026-04-01T20-22-32.json


Silver transform for NASA EONET data.

In [17]:
import json
import csv
import glob
from collections import defaultdict



In [35]:

def transform_eonet():
    # 1. Find the latest bronze file
    bronze_files = sorted(glob.glob("data/bronze/eonet/eonet_events_*.json"))
    if not bronze_files:
        # Check if maybe the naming is slightly different
        bronze_files = sorted(glob.glob("data/bronze/eonet/*.json"))
        
    if not bronze_files:
        raise FileNotFoundError("No EONET bronze files found in data/bronze/eonet/")

    latest = bronze_files[-1]

    with open(latest, "r") as f:
        raw = json.load(f)

    # 2. Count events per date per category
    daily = defaultdict(lambda: {"event_count": 0, "wildfire_count": 0, "storm_count": 0})

    for event in raw.get("events", []):
        category = event["categories"][0]["id"] if event.get("categories") else "unknown"
        dates = []
        for geo in event.get("geometry", []):
            if geo.get("date"):
                dates.append(geo["date"][:10])

        if not dates:
            continue

        # Filter to only 2024 dates (Adjust to 2026 if your data is current)
        dates_filtered = [d for d in dates if d.startswith("2024")]
        if not dates_filtered:
            continue

        event_date = min(dates_filtered)
        daily[event_date]["event_count"] += 1
        if category == "wildfires":
            daily[event_date]["wildfire_count"] += 1
        elif category == "severeStorms":
            daily[event_date]["storm_count"] += 1

    # 3. Create the silver directory if it doesn't exist (THIS FIXES YOUR ERROR)
    out_dir = "data/silver"
    os.makedirs(out_dir, exist_ok=True)
    
    out_path = os.path.join(out_dir, "eonet_daily_counts.csv")

    # 4. Write CSV sorted by date
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["date", "event_count", "wildfire_count", "storm_count"])
        for date in sorted(daily.keys()):
            row = daily[date]
            writer.writerow([date, row["event_count"], row["wildfire_count"], row["storm_count"]])

    print(f"Success! {len(daily)} rows saved to {out_path}")

if __name__ == "__main__":
    transform_eonet()

Success! 304 rows saved to data/silver\eonet_daily_counts.csv


Silver transform for Open-Meteo historical weather data

In [ ]:
import json
import csv
import glob
import os


In [38]:
def transform_weather():
    # 1. Find the latest bronze file
    bronze_files = sorted(glob.glob("data/bronze/weather/la_weather_*.json"))
    if not bronze_files:
        raise FileNotFoundError("No weather bronze files found in data/bronze/weather/")

    latest = bronze_files[-1]

    with open(latest, "r") as f:
        raw = json.load(f)

    # 2. Extract daily data
    daily = raw["daily"]
    dates = daily["time"]
    temp_max = daily["temperature_2m_max"]
    temp_min = daily["temperature_2m_min"]
    precip = daily["precipitation_sum"]
    wind = daily["windspeed_10m_max"]

    # 3. Create the silver directory if it doesn't exist
    out_dir = "data/silver"
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "la_weather_daily.csv")

    # 4. Write CSV
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["date", "temp_max", "temp_min", "precipitation_sum", "windspeed_max"])
        for i in range(len(dates)):
            writer.writerow([
                dates[i],
                temp_max[i] if temp_max[i] is not None else "",
                temp_min[i] if temp_min[i] is not None else "",
                precip[i] if precip[i] is not None else 0.0,
                wind[i] if wind[i] is not None else ""
            ])

    # 5. Success Message (Replaces the broken /tmp/ file logic)
    print(f"Success! {len(dates)} rows saved to {out_path}")

if __name__ == "__main__":
    transform_weather()

Success! 366 rows saved to data/silver\la_weather_daily.csv


Gold layer:
Join EONET daily event counts with LA weather data using DuckDB.

In [39]:
import duckdb

In [41]:
import os

In [43]:
def build_gold():
    # 1. Create the gold directory first!
    gold_dir = "data/gold"
    os.makedirs(gold_dir, exist_ok=True)
    
    # 2. Define the SQL logic
    sql = """
    COPY (
        WITH weather AS (
            SELECT * FROM read_csv_auto('data/silver/la_weather_daily.csv')
        ),
        eonet AS (
            SELECT * FROM read_csv_auto('data/silver/eonet_daily_counts.csv')
        ),
        joined AS (
            SELECT
                w.date,
                w.temp_max,
                w.temp_min,
                w.precipitation_sum,
                w.windspeed_max,
                COALESCE(e.event_count, 0) AS event_count,
                COALESCE(e.wildfire_count, 0) AS wildfire_count,
                COALESCE(e.storm_count, 0) AS storm_count
            FROM weather w
            LEFT JOIN eonet e ON w.date = e.date
        ),
        with_median AS (
            SELECT *,
            MEDIAN(windspeed_max) OVER () AS wind_median
            FROM joined
        )
        SELECT
            date,
            temp_max,
            temp_min,
            precipitation_sum,
            windspeed_max,
            event_count,
            wildfire_count,
            storm_count,
            CASE WHEN precipitation_sum > 0 THEN 1 ELSE 0 END AS rainy_day,
            CASE WHEN event_count > 0 THEN 1 ELSE 0 END AS event_day,
            CASE WHEN windspeed_max > wind_median THEN 1 ELSE 0 END AS high_wind_day,
            CASE
                WHEN MONTH(date) IN (12, 1, 2) THEN 'Winter'
                WHEN MONTH(date) IN (3, 4, 5) THEN 'Spring'
                WHEN MONTH(date) IN (6, 7, 8) THEN 'Summer'
                ELSE 'Fall'
            END AS season,
            -- Note: DuckDB DAYOFWEEK usually returns 0-6 (Sunday-Saturday)
            CASE WHEN DAYOFWEEK(date) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend
        FROM with_median
        ORDER BY date
    ) TO 'data/gold/nasa_weather_daily.csv' (HEADER, DELIMITER ',');
    """

    # 3. Execute the SQL
    duckdb.execute(sql)
    
    # 4. Verify and print success
    gold_file = 'data/gold/nasa_weather_daily.csv'
    count = duckdb.query(f"SELECT COUNT(*) FROM read_csv_auto('{gold_file}')").fetchone()[0]
    print(f"Success! Gold Table Created: {count} rows -> {gold_file}")

if __name__ == "__main__":
    build_gold()

Success! Gold Table Created: 366 rows -> data/gold/nasa_weather_daily.csv


Part 2:
Streamlit app that reads  Gold dataset


2026-04-02 12:34:36.691 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 12:34:36.694 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 12:34:37.947 
  command:

    streamlit run c:\Users\ALI COMPUTER\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-02 12:34:37.948 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 12:34:37.950 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 12:34:37.954 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-02 12:34:37.955 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored wh

DeltaGenerator()